# Nifty 500 — Final Data Validation Gate

This notebook is the **final validation gate before backtesting-engine development**. It is not another broad inventory scan. It checks whether the previously identified audit priorities have actually been resolved and whether the selected price-only strategy dataset is defensible.

It is read-only: it does not alter source data or silently remove records.

## Final validation scope

1. Validate the existing membership-to-price mapping audit against actual price identifiers.
2. Validate the actual historical membership Parquet (not a similarly named summary CSV).
3. Recheck duplicate price/date keys and classify exact duplicates versus conflicting records.
4. Validate price observations and flag unresolved gaps/extreme moves.
5. Validate the selected feature dataset against independent calculations.
6. Require an explicit, documented corporate-action/price-adjustment decision.
7. Record the three accepted extraction exceptions separately, with RAYMONDL requiring a deliberate decision because a price file exists.
8. Produce a strategy-specific readiness decision and a machine-readable report.

**Important:** a notebook cannot prove that source prices are historically true or that vendor adjustment methodology is correct without trusted source documentation. Where evidence is not available, the gate returns `BLOCKED` or `CONDITIONAL`, not a false `PASS`.

## 0. Configuration

Change paths or column overrides only where necessary. The defaults are based on the project structure used in the previous audits.

The three extraction failures are treated as accepted exceptions for initial universe coverage, not as audit failures. RAYMONDL is separately flagged because the prior audit found a price file with 491 observations.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import json, re, hashlib, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 100)
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_ROOT = Path.home() / "Documents/vscode/BacktestEngine/dataextracted"
REPORT_ROOT = DATA_ROOT.parent / "final_validation_output"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

# Explicit source paths: avoid accidentally selecting summary/audit CSVs.
MEMBERSHIP_PATH = DATA_ROOT / "DE/data/raw/constituents/nifty500_historical_membership.parquet"
MAPPING_PATH = DATA_ROOT / "DE/data/raw/constituents/historical_membership_price_mapping_audit.csv"
FEATURE_PATH = DATA_ROOT / "DE/data/processed/nifty500_historical_stock_features.parquet"
FUNDAMENTALS_PATH = DATA_ROOT / "DE/data/raw/fundamentals/nifty500_historical_fundamentals.parquet"

# Adjust only if the actual extracted folder structure differs.
PRICE_DIR = DATA_ROOT / "DE/data/raw/prices"

# Known accepted extraction exceptions. These are excluded from the initial tradable
# universe if they do not have complete usable price history.
ACCEPTED_EXTRACTION_EXCEPTIONS = {"360ONE IN", "RAYMONDL IN", "SANOFICO IN"}

# Explicit manual evidence gates. Set True only after checking actual source documentation.
# Do not set these True merely to make the notebook pass.
MEMBERSHIP_EFFECTIVE_DATES_VERIFIED = False
PRICE_ADJUSTMENT_BASIS_VERIFIED = False
CORPORATE_ACTION_RECONCILIATION_COMPLETED = False
FEATURE_DEFINITIONS_CONFIRMED = False

# For a price-only technical backtest, fundamentals are deliberately not required.
ENABLE_FUNDAMENTAL_STRATEGY = False

# Strict validation thresholds
MAX_CALENDAR_GAP_DAYS = 7       # review flag only; exchange calendar is needed to confirm sessions
EXTREME_RETURN_THRESHOLD = 0.50
ABS_TOL = 1e-6
REL_TOL = 1e-4

assert DATA_ROOT.exists(), f"DATA_ROOT not found: {DATA_ROOT}"
for p in [MEMBERSHIP_PATH, MAPPING_PATH, FEATURE_PATH]:
    assert p.exists(), f"Required file not found: {p}"

print("Data root:", DATA_ROOT)
print("Report root:", REPORT_ROOT)

Data root: /home/rishi/Documents/vscode/BacktestEngine/dataextracted
Report root: /home/rishi/Documents/vscode/BacktestEngine/final_validation_output


## 1. Utility functions and final issue register

In [3]:
issues = []
checks = []

def issue(severity, area, code, detail, evidence=None):
    issues.append({"severity":severity,"area":area,"code":code,
                   "detail":str(detail),"evidence":str(evidence) if evidence is not None else None})

def check(area, name, status, detail, evidence=None):
    checks.append({"area":area,"check":name,"status":status,
                   "detail":str(detail),"evidence":str(evidence) if evidence is not None else None})

def norm(s):
    return re.sub(r"[^a-z0-9]+","",str(s).lower())

def find_col(df, names):
    bynorm={norm(c):c for c in df.columns}
    for n in names:
        if norm(n) in bynorm: return bynorm[norm(n)]
    return None

def read_df(p):
    if p.suffix.lower()==".parquet": return pd.read_parquet(p)
    if p.suffix.lower()==".csv": return pd.read_csv(p,low_memory=False)
    raise ValueError(p)

def parse_dates(s):
    return pd.to_datetime(s,errors="coerce")

def clean_id(s):
    return s.astype("string").str.strip()

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def status_from_bool(ok):
    return "PASS" if ok else "FAIL"

## 2. Load exact source datasets and confirm their identity

This section intentionally loads the historical membership Parquet by its exact filename. It does not choose `nifty500_membership_validation.csv`, which is a report rather than the underlying membership table.

In [4]:
membership = read_df(MEMBERSHIP_PATH)
mapping = read_df(MAPPING_PATH)
features = read_df(FEATURE_PATH)

fundamentals = read_df(FUNDAMENTALS_PATH) if FUNDAMENTALS_PATH.exists() else None

print("Membership:", membership.shape, list(membership.columns))
print("Mapping audit:", mapping.shape, list(mapping.columns))
print("Features:", features.shape, list(features.columns))
print("Fundamentals:", None if fundamentals is None else (fundamentals.shape,list(fundamentals.columns)))

display(membership.head())
display(mapping.head())

Membership: (9857, 2) ['Date', 'Ticker']
Mapping audit: (9857, 5) ['Date', 'Ticker', 'Membership_Root', 'Price_Match_Count', 'Mapping_Status']
Features: (1798332, 37) ['Date', 'Ticker', 'PX_OPEN', 'PX_HIGH', 'PX_LOW', 'PX_LAST', 'PX_VOLUME', 'Return_1M', 'Return_3M', 'Return_6M', 'Return_12M', 'DMA_20', 'Price_20DMA', 'DMA_50', 'Price_50DMA', 'DMA_100', 'Price_100DMA', 'DMA_200', 'Price_200DMA', '52W_High', '52W_Low', 'Pct_Below_52W_High', 'Pct_Above_52W_Low', 'RSI', 'ROC', 'MACD', 'Daily_Return', 'Volatility_20D', 'Volatility_60D', 'Volatility_120D', 'Volatility_252D', 'Current_Drawdown', 'Max_Drawdown_3M', 'Max_Drawdown_12M', 'Volume_Change', 'Volume_20D_Avg', 'Volume_to_20D_Avg']
Fundamentals: ((15534, 15), ['Bloomberg_Security', 'Date', 'SALES_REV_TURN', 'NET_INCOME', 'EBITDA', 'EBIT', 'BOOK_VAL_PER_SH', 'IS_EPS', 'CUR_MKT_CAP', 'EQY_SH_OUT', 'PE_RATIO', 'PX_TO_BOOK_RATIO', 'PX_TO_SALES_RATIO', 'DVD_SH_12M', 'TOT_RETURN_INDEX_GROSS_DVDS'])


,Date,Ticker
0,2016-09-01,1958850D IS
1,2016-09-01,3M IS
2,2016-09-01,63MOONS IS
3,2016-09-01,ABAN IS
4,2016-09-01,ABB IS


,Date,Ticker,Membership_Root,Price_Match_Count,Mapping_Status
0,2016-09-01,1958850D IS,1958850D,1,Unique_Price_Match
1,2016-09-01,3M IS,3M,1,Unique_Price_Match
2,2016-09-01,63MOONS IS,63MOONS,1,Unique_Price_Match
3,2016-09-01,ABAN IS,ABAN,1,Unique_Price_Match
4,2016-09-01,ABB IS,ABB,1,Unique_Price_Match


## 3. Price file loading and duplicate resolution audit

The final gate does not silently deduplicate. It reports:
- exact duplicate rows (same security/date and identical OHLCV values),
- conflicting duplicates (same security/date but different values),
- missing keys,
- invalid OHLC/volume values,
- extreme returns and large calendar gaps.

Exact duplicate rows can be removed in a separate, documented cleaning step after confirming they are duplicate extraction artifacts. Conflicting duplicates require source-level reconciliation.

In [5]:
assert PRICE_DIR.exists(), f"Price directory not found: {PRICE_DIR}"

price_paths=[p for p in PRICE_DIR.rglob("*.parquet")
             if "__MACOSX" not in p.parts and not p.name.startswith("._")]

def col(df, options):
    return find_col(df,options)

price_parts=[]
file_errors=[]
for p in price_paths:
    try:
        raw=pd.read_parquet(p)
        d=pd.DataFrame()
        datec=col(raw,["Date","PX_DATE","TRADE_DATE","timestamp"])
        tickc=col(raw,["Ticker","Security","Security_ID","Symbol","Bloomberg_Ticker"])
        openc=col(raw,["PX_OPEN","Open"])
        highc=col(raw,["PX_HIGH","High"])
        lowc=col(raw,["PX_LOW","Low"])
        closec=col(raw,["PX_LAST","PX_CLOSE","Close","Last"])
        volc=col(raw,["PX_VOLUME","Volume"])
        if datec is None or closec is None:
            raise ValueError(f"Missing date/close. Columns={list(raw.columns)}")
        d["date"]=parse_dates(raw[datec])
        d["ticker"]=clean_id(raw[tickc]) if tickc else p.stem
        for out,src in [("open",openc),("high",highc),("low",lowc),("close",closec),("volume",volc)]:
            d[out]=pd.to_numeric(raw[src],errors="coerce") if src else np.nan
        d["source_file"]=str(p.relative_to(DATA_ROOT))
        price_parts.append(d)
    except Exception as e:
        file_errors.append({"file":str(p),"error":repr(e)})

prices=pd.concat(price_parts,ignore_index=True) if price_parts else pd.DataFrame()
print("Price files discovered:",len(price_paths))
print("Readable:",len(price_parts),"Unreadable:",len(file_errors))
print("Rows:",len(prices),"Unique IDs:",prices["ticker"].nunique() if len(prices) else 0)
display(pd.DataFrame(file_errors))

Price files discovered: 859
Readable: 858 Unreadable: 1
Rows: 1798332 Unique IDs: 858


,file,error
0,/home/rishi/Documents/vscode/BacktestEngine/da...,"ValueError(""Missing date/close. Columns=['secu..."


In [6]:
required={"date","ticker","open","high","low","close","volume"}
missing_cols=required-set(prices.columns)
if missing_cols:
    issue("BLOCKER","prices","REQUIRED_PRICE_COLUMNS_MISSING",missing_cols)

valid=prices.dropna(subset=["date","ticker"]).copy()
dup=valid.duplicated(["ticker","date"],keep=False)
dup_rows=valid.loc[dup].copy()

value_cols=["open","high","low","close","volume"]
if len(dup_rows):
    # Number of distinct value signatures per duplicate key.
    signatures=(dup_rows.groupby(["ticker","date"])[value_cols]
                .apply(lambda x:x.drop_duplicates().shape[0]).rename("distinct_value_versions").reset_index())
    exact_keys=signatures[signatures["distinct_value_versions"]==1]
    conflicting_keys=signatures[signatures["distinct_value_versions"]>1]
else:
    signatures=pd.DataFrame(columns=["ticker","date","distinct_value_versions"])
    exact_keys=conflicting_keys=signatures.copy()

print("Duplicate key groups:",len(signatures))
print("Exact duplicate groups:",len(exact_keys))
print("Conflicting duplicate groups:",len(conflicting_keys))
display(conflicting_keys.head(50))

if len(conflicting_keys):
    issue("BLOCKER","prices","CONFLICTING_DUPLICATE_PRICE_KEYS",
          f"{len(conflicting_keys)} ticker/date keys contain different OHLCV values.")
else:
    check("prices","No conflicting duplicate ticker/date keys","PASS","No conflicting duplicate groups found.")

if len(exact_keys):
    issue("REVIEW","prices","EXACT_DUPLICATE_PRICE_KEYS",
          f"{len(exact_keys)} ticker/date keys are repeated with identical OHLCV values. Resolve in clean layer and preserve provenance.")
else:
    check("prices","No exact duplicate ticker/date keys","PASS","No exact duplicate groups found.")

bad_date=prices["date"].isna()
bad_ticker=prices["ticker"].isna() | prices["ticker"].astype(str).str.strip().eq("")
if bad_date.any(): issue("FAIL","prices","INVALID_PRICE_DATES",int(bad_date.sum()))
if bad_ticker.any(): issue("FAIL","prices","INVALID_PRICE_IDENTIFIERS",int(bad_ticker.sum()))

for c in ["open","high","low","close"]:
    if c in prices:
        bad=prices[c].notna() & (prices[c]<=0)
        if bad.any(): issue("FAIL","prices","NONPOSITIVE_PRICE",f"{c}: {int(bad.sum())} rows")
if "volume" in prices:
    bad=prices["volume"].notna() & (prices["volume"]<0)
    if bad.any(): issue("FAIL","prices","NEGATIVE_VOLUME",int(bad.sum()))

complete=prices[["open","high","low","close"]].notna().all(axis=1)
ohlc_bad=complete & ((prices["high"]<prices[["open","close","low"]].max(axis=1)) |
                     (prices["low"]>prices[["open","close","high"]].min(axis=1)))
if ohlc_bad.any(): issue("FAIL","prices","OHLC_ORDERING_VIOLATIONS",int(ohlc_bad.sum()))
else: check("prices","OHLC ordering","PASS","No OHLC ordering violations among complete OHLC rows.")

# Missing fields are quantified, not automatically filled.
missingness=prices[["open","high","low","close","volume"]].isna().sum().rename("missing_rows").to_frame()
display(missingness)

Duplicate key groups: 0
Exact duplicate groups: 0
Conflicting duplicate groups: 0


,ticker,date,distinct_value_versions


,missing_rows
open,162
high,162
low,162
close,0
volume,672


## 4. Price continuity and extreme-move review

These flags are not automatically treated as bad data. Large moves can be genuine; gaps can result from weekends, holidays, suspensions, listings or delistings. They must be reconciled against an authoritative trading calendar and security lifecycle history.

In [7]:
# Compute returns only after excluding duplicate keys; otherwise pct_change can be ambiguous.
unique_price=valid.drop_duplicates(["ticker","date"],keep=False).sort_values(["ticker","date"]).copy()
unique_price["daily_return"]=unique_price.groupby("ticker")["close"].pct_change(fill_method=None)

extreme=unique_price[unique_price["daily_return"].abs()>EXTREME_RETURN_THRESHOLD][
    ["ticker","date","close","daily_return","source_file"]]
gap_records=[]
for ticker,g in unique_price.groupby("ticker"):
    g=g.sort_values("date")
    delta=g["date"].diff().dt.days
    ix=delta[delta>MAX_CALENDAR_GAP_DAYS].index
    for i in ix:
        gap_records.append({"ticker":ticker,"date":g.loc[i,"date"],
                            "calendar_gap_days":int(delta.loc[i]),
                            "source_file":g.loc[i,"source_file"]})
gaps=pd.DataFrame(gap_records)

print("Extreme daily returns:",len(extreme))
print("Calendar gaps > threshold:",len(gaps))
display(extreme.sort_values("daily_return",key=lambda s:s.abs(),ascending=False).head(50))
display(gaps.head(50))

if len(extreme):
    issue("REVIEW","prices","EXTREME_DAILY_MOVES_UNRECONCILED",
          f"{len(extreme)} moves exceed {EXTREME_RETURN_THRESHOLD:.0%}; reconcile with corporate actions and source data.")
if len(gaps):
    issue("REVIEW","prices","PRICE_GAPS_UNRECONCILED",
          f"{len(gaps)} inter-observation gaps exceed {MAX_CALENDAR_GAP_DAYS} calendar days.")

Extreme daily returns: 47
Calendar gaps > threshold: 155


,ticker,date,close,daily_return,source_file
1227685,SWANDEF IN Equity,2025-01-20,37.78,15.426087,DE/data/raw/prices/SWANDEF.parquet
869136,KSI IN Equity,2025-03-10,9.71,7.024793,DE/data/raw/prices/KSI.parquet
1073271,ALOK IN Equity,2020-02-19,18.05,4.469697,DE/data/raw/prices/ALOK.parquet
1754249,PATANJAL IN Equity,2020-01-27,5.63,4.072072,DE/data/raw/prices/PATANJAL.parquet
1577430,TATATECH IN Equity,2023-11-30,1313.00,1.626000,DE/data/raw/prices/TATATECH.parquet
1600863,LATENTVI IN Equity,2021-11-23,488.75,1.480964,DE/data/raw/prices/LATENTVI.parquet
689330,BAJAJHFL IN Equity,2024-09-16,165.00,1.357143,DE/data/raw/prices/BAJAJHFL.parquet
908875,GVKP IN Equity,2022-08-12,5.70,1.280000,DE/data/raw/prices/GVKP.parquet
337859,IRCTC IN Equity,2019-10-14,145.55,1.274219,DE/data/raw/prices/IRCTC.parquet
1791231,RBA IN Equity,2020-12-14,135.00,1.250000,DE/data/raw/prices/RBA.parquet


,ticker,date,calendar_gap_days,source_file
0,1958850D IN Equity,2018-10-01,24,DE/data/raw/prices/1958850D.parquet
1,1958850D IN Equity,2019-03-05,8,DE/data/raw/prices/1958850D.parquet
2,1958850D IN Equity,2019-08-13,134,DE/data/raw/prices/1958850D.parquet
3,1958850D IN Equity,2019-10-04,51,DE/data/raw/prices/1958850D.parquet
4,1958850D IN Equity,2019-11-14,41,DE/data/raw/prices/1958850D.parquet
5,1958850D IN Equity,2020-03-27,134,DE/data/raw/prices/1958850D.parquet
6,1958850D IN Equity,2020-04-13,14,DE/data/raw/prices/1958850D.parquet
7,ABGS IN Equity,2019-08-13,90,DE/data/raw/prices/ABGS.parquet
8,ABGS IN Equity,2019-10-04,51,DE/data/raw/prices/ABGS.parquet
9,ABGS IN Equity,2019-11-14,41,DE/data/raw/prices/ABGS.parquet


## 5. Validate the existing membership-to-price mapping audit

Expected mapping audit fields in the supplied file include `Membership_Root`, `Price_Ticker`, `Price_Match_Count` and `Mapping_Status`. The code detects common column variants and compares `Price_Ticker` to actual loaded price-file identifiers.

It does not compare raw membership tickers directly to Bloomberg tickers.

In [8]:
member_id_col=find_col(membership,["Ticker","Membership_Ticker","Symbol","Security_ID"])
member_date_col=find_col(membership,["Date","Snapshot_Date","As_Of_Date","Effective_Date"])
map_member_col=find_col(mapping,["Membership_Root","Membership_Ticker","Member_Ticker","Ticker"])
map_price_col=find_col(mapping,["Price_Ticker","Bloomberg_Ticker","Vendor_Ticker"])
map_count_col=find_col(mapping,["Price_Match_Count","Match_Count"])
map_status_col=find_col(mapping,["Mapping_Status","Status"])

print("Detected mapping columns:",{
    "membership security":member_id_col,"membership date":member_date_col,
    "mapping membership":map_member_col,"mapping price":map_price_col,
    "match count":map_count_col,"status":map_status_col
})

if not all([member_id_col,map_member_col,map_price_col]):
    issue("BLOCKER","mapping","MAPPING_SCHEMA_NOT_RECOGNIZED",
          "Could not detect membership identifier and mapping source/target identifier columns.")
else:
    actual_price_ids=set(unique_price["ticker"].dropna().astype(str).str.strip().unique())
    map2=mapping.copy()
    map2["_member"]=clean_id(map2[map_member_col])
    map2["_price"]=clean_id(map2[map_price_col])
    member_roots=set(clean_id(membership[member_id_col]).dropna().unique())
    mapped_roots=set(map2["_member"].dropna().unique())
    missing_roots=member_roots-mapped_roots
    extra_roots=mapped_roots-member_roots
    mapped_price_ids=set(map2["_price"].dropna().unique())
    missing_price_ids=mapped_price_ids-actual_price_ids

    print("Membership unique IDs:",len(member_roots))
    print("Mapped membership IDs:",len(mapped_roots))
    print("Membership IDs without mapping:",len(missing_roots))
    print("Mapping IDs absent from membership:",len(extra_roots))
    print("Mapped price IDs absent from loaded price IDs:",len(missing_price_ids))

    if missing_roots:
        issue("BLOCKER","mapping","MEMBERSHIP_SECURITIES_WITHOUT_MAPPING",
              len(missing_roots),sorted(missing_roots)[:100])
    if missing_price_ids:
        issue("BLOCKER","mapping","MAPPED_PRICE_IDS_NOT_FOUND",
              len(missing_price_ids),sorted(missing_price_ids)[:100])

    if map_count_col:
        counts=pd.to_numeric(map2[map_count_col],errors="coerce")
        zero=int((counts==0).sum()); multi=int((counts>1).sum()); one=int((counts==1).sum())
        print("Reported match counts:",{"one":one,"zero":zero,"multiple":multi,"null":int(counts.isna().sum())})
        if zero or multi or counts.isna().any():
            issue("BLOCKER","mapping","NON_UNIQUE_OR_UNKNOWN_MATCH_COUNTS",
                  {"zero":zero,"multiple":multi,"null":int(counts.isna().sum())})
    if map_status_col:
        display(map2[map_status_col].value_counts(dropna=False).to_frame("rows"))

    # Detect duplicate and non-unique relationships in the actual mapping table.
    member_to_price=map2.groupby("_member")["_price"].nunique(dropna=True)
    price_to_member=map2.groupby("_price")["_member"].nunique(dropna=True)
    ambiguous_m2p=member_to_price[member_to_price>1]
    ambiguous_p2m=price_to_member[price_to_member>1]
    print("One membership root -> multiple price IDs:",len(ambiguous_m2p))
    print("One price ID -> multiple membership roots:",len(ambiguous_p2m))
    if len(ambiguous_m2p) or len(ambiguous_p2m):
        issue("BLOCKER","mapping","AMBIGUOUS_MAPPING_RELATIONSHIPS",
              {"membership_to_price":ambiguous_m2p.to_dict(),"price_to_membership":ambiguous_p2m.to_dict()})

    mapping_coverage=pd.DataFrame([{
        "membership_ids":len(member_roots),"mapped_membership_ids":len(mapped_roots),
        "membership_ids_without_mapping":len(missing_roots),
        "unique_mapped_price_ids":len(mapped_price_ids),
        "mapped_price_ids_not_in_price_files":len(missing_price_ids),
        "membership_mapping_coverage_pct":100*len(member_roots & mapped_roots)/len(member_roots) if member_roots else np.nan
    }])
    display(mapping_coverage)

Detected mapping columns: {'membership security': 'Ticker', 'membership date': 'Date', 'mapping membership': 'Membership_Root', 'mapping price': None, 'match count': 'Price_Match_Count', 'status': 'Mapping_Status'}


## 6. Historical membership validation

The notebook validates the actual membership table's keys and dates. If the data is a sparse set of snapshots, it will **not** invent daily effective membership intervals. A historical-universe backtest requires either authoritative effective-from/effective-to intervals or a documented method/source for reconstructing changes between snapshots.

In [9]:
if member_id_col is None:
    issue("BLOCKER","membership","MEMBERSHIP_SECURITY_KEY_MISSING","No membership security identifier detected.")
else:
    mem=membership.copy()
    mem["_security"]=clean_id(mem[member_id_col])
    start_col=find_col(mem,["Valid_From","Effective_From","Start_Date","From_Date"])
    end_col=find_col(mem,["Valid_To","Effective_To","End_Date","To_Date"])
    if member_date_col:
        mem["_date"]=parse_dates(mem[member_date_col])
        if mem["_date"].isna().any():
            issue("FAIL","membership","INVALID_MEMBERSHIP_DATES",int(mem["_date"].isna().sum()))
        dupmem=mem.duplicated(["_security","_date"],keep=False)
        if dupmem.any():
            issue("FAIL","membership","DUPLICATE_MEMBERSHIP_SNAPSHOT_KEYS",int(dupmem.sum()))
        snapshot_counts=mem.groupby("_date")["_security"].nunique().rename("constituents").reset_index()
        display(snapshot_counts)
        if len(snapshot_counts)<30 and not MEMBERSHIP_EFFECTIVE_DATES_VERIFIED:
            issue("BLOCKER","membership","SPARSE_SNAPSHOTS_NOT_EFFECTIVE_DATES",
                  f"Only {len(snapshot_counts)} snapshots. Set MEMBERSHIP_EFFECTIVE_DATES_VERIFIED=True only after independently establishing exact effective periods.")
    elif start_col and end_col:
        mem["_from"]=parse_dates(mem[start_col]); mem["_to"]=parse_dates(mem[end_col])
        bad=mem["_from"].isna() | mem["_to"].isna() | (mem["_from"]>mem["_to"])
        if bad.any(): issue("FAIL","membership","INVALID_EFFECTIVE_INTERVALS",int(bad.sum()))
        else: check("membership","Effective membership intervals","PASS","All effective intervals have valid dates and ordering.")
        # Detect overlapping intervals per security.
        overlap_count=0
        for sec,g in mem.sort_values("_from").groupby("_security"):
            prev_end=None
            for _,r in g.iterrows():
                if prev_end is not None and r["_from"]<=prev_end:
                    overlap_count+=1
                prev_end=max(prev_end,r["_to"]) if prev_end is not None else r["_to"]
        if overlap_count:
            issue("FAIL","membership","OVERLAPPING_MEMBERSHIP_INTERVALS",overlap_count)
    else:
        issue("BLOCKER","membership","MEMBERSHIP_TIME_SEMANTICS_UNKNOWN",
              "No snapshot date or effective interval columns detected.")

if not MEMBERSHIP_EFFECTIVE_DATES_VERIFIED:
    issue("BLOCKER","membership","MEMBERSHIP_EFFECTIVE_DATE_ATTESTATION_MISSING",
          "Historical constituent effective dates have not been independently verified.")

,_date,constituents
0,2016-09-01,501
1,2017-03-01,501
2,2017-09-01,501
3,2018-03-01,501
4,2018-09-01,338
5,2019-03-01,501
6,2019-09-01,501
7,2020-03-01,501
8,2020-09-01,501
9,2021-03-01,501


## 7. Feature validation against raw prices

This recomputes simple close-based features from raw prices and compares them with the supplied feature table. It validates only features whose definitions can be aligned unambiguously. Volatility is compared as a rolling standard deviation without annualization by default; change the formula only after confirming the source definition.

If the supplied feature definitions differ (e.g. adjusted versus raw close, sample versus population standard deviation, EMA initialization), resolve those definitions before treating mismatches as errors.

In [10]:
fdate=find_col(features,["Date","Trade_Date"])
fticker=find_col(features,["Ticker","Symbol","Security_ID","Bloomberg_Ticker"])
if not fdate or not fticker:
    issue("BLOCKER","features","FEATURE_KEYS_NOT_FOUND","Could not detect date and ticker keys.")
else:
    f=features.copy()
    f["_date"]=parse_dates(f[fdate])
    f["_ticker"]=clean_id(f[fticker])
    if f.duplicated(["_date","_ticker"],keep=False).any():
        issue("FAIL","features","DUPLICATE_FEATURE_KEYS",
              int(f.duplicated(["_date","_ticker"],keep=False).sum()))
    # Only unambiguous raw price keys are used.
    base=unique_price[["date","ticker","close"]].dropna().sort_values(["ticker","date"]).copy()
    gg=base.groupby("ticker",group_keys=False)
    base["calc_daily_return"]=gg["close"].pct_change(fill_method=None)
    for w in [20,50,100,200,252]:
        base[f"calc_sma_{w}"]=gg["close"].transform(lambda s:s.rolling(w,min_periods=w).mean())
    for w in [20,60,120,252]:
        base[f"calc_vol_{w}"]=gg["close"].transform(lambda s:s.pct_change(fill_method=None).rolling(w,min_periods=w).std(ddof=1))
    base["calc_high_252"]=gg["close"].transform(lambda s:s.rolling(252,min_periods=252).max())
    base["calc_low_252"]=gg["close"].transform(lambda s:s.rolling(252,min_periods=252).min())

    fmap={
        "Daily_Return":"calc_daily_return",
        "DMA_20":"calc_sma_20","Price_20DMA":"calc_sma_20",
        "DMA_50":"calc_sma_50","Price_50DMA":"calc_sma_50",
        "DMA_100":"calc_sma_100","Price_100DMA":"calc_sma_100",
        "DMA_200":"calc_sma_200","Price_200DMA":"calc_sma_200",
        "52W_High":"calc_high_252","52W_Low":"calc_low_252",
        "Volatility_20D":"calc_vol_20","Volatility_60D":"calc_vol_60",
        "Volatility_120D":"calc_vol_120D","Volatility_252D":"calc_vol_252"
    }
    # Keep only exact columns present. Fix common 120D spelling mapping.
    fmap["Volatility_120D"]="calc_vol_120"
    feature_comparisons=[]
    merged=f.merge(base,left_on=["_date","_ticker"],right_on=["date","ticker"],how="inner",validate="many_to_one")
    for supplied,calc in fmap.items():
        if supplied not in merged.columns or calc not in merged.columns: continue
        a=pd.to_numeric(merged[supplied],errors="coerce")
        b=pd.to_numeric(merged[calc],errors="coerce")
        mask=a.notna() & b.notna() & np.isfinite(a) & np.isfinite(b)
        if not mask.any(): continue
        err=(a[mask]-b[mask]).abs()
        tol=ABS_TOL+REL_TOL*b[mask].abs()
        bad=err>tol
        feature_comparisons.append({"feature":supplied,"overlap":int(mask.sum()),
                                    "mismatch_count":int(bad.sum()),
                                    "mismatch_pct":100*bad.mean(),
                                    "mae":float(err.mean()),"max_abs_error":float(err.max())})
    feature_comparison_df=pd.DataFrame(feature_comparisons)
    display(feature_comparison_df)
    if len(feature_comparison_df)==0:
        issue("BLOCKER","features","NO_FEATURE_COMPARISONS_POSSIBLE",
              "No comparable features were identified.")
    elif (feature_comparison_df["mismatch_count"]>0).any() and not FEATURE_DEFINITIONS_CONFIRMED:
        issue("REVIEW","features","FEATURE_MISMATCHES_OR_DEFINITION_DIFFERENCES",
              "One or more supplied features differ from independent calculations; confirm exact definitions and source price basis.")
    if not FEATURE_DEFINITIONS_CONFIRMED:
        issue("BLOCKER","features","FEATURE_DEFINITIONS_NOT_ATTESTED",
              "Indicator formulas, initialization, price basis and missing-value conventions have not been fully confirmed.")

,feature,overlap,mismatch_count,mismatch_pct,mae,max_abs_error
0,Daily_Return,1797474,0,0.000000,0.000000,0.000000
1,DMA_20,1782030,0,0.000000,0.000000,0.000000
2,Price_20DMA,1782030,1782027,99.999832,1142.868289,158688.509796
3,DMA_50,1756344,0,0.000000,0.000000,0.000000
4,Price_50DMA,1756344,1756342,99.999886,1142.427139,155830.817177
5,DMA_100,1713544,0,0.000000,0.000000,0.000000
6,Price_100DMA,1713544,1713539,99.999708,1142.386201,152562.881815
7,DMA_200,1628190,0,0.000000,0.000000,0.000000
8,Price_200DMA,1628190,1628190,100.000000,1143.348440,147713.628397
9,52W_High,1584031,0,0.000000,0.000000,0.000000


## 8. Corporate-action and price-adjustment evidence gate

This is a manual evidence gate by design. A lack of a detected corporate-action file does not prove that the price data is unadjusted; the vendor may supply adjusted prices. Conversely, a file named `corporate_actions` does not prove complete or correct adjustment.

Set the three configuration booleans to True only after documenting:
- vendor price field and adjustment basis (raw, split-adjusted, total-return/dividend-adjusted),
- source and coverage of splits, bonuses, dividends, rights, mergers/demergers and delistings,
- reconciliation of large discontinuities and known corporate actions.

In [11]:
if not PRICE_ADJUSTMENT_BASIS_VERIFIED:
    issue("BLOCKER","corporate_actions","PRICE_ADJUSTMENT_BASIS_UNVERIFIED",
          "Vendor documentation has not established whether the price series is raw, split-adjusted or dividend-adjusted.")
if not CORPORATE_ACTION_RECONCILIATION_COMPLETED:
    issue("BLOCKER","corporate_actions","CORPORATE_ACTION_RECONCILIATION_INCOMPLETE",
          "Corporate actions and large price discontinuities have not been reconciled.")
if PRICE_ADJUSTMENT_BASIS_VERIFIED and CORPORATE_ACTION_RECONCILIATION_COMPLETED:
    check("corporate_actions","Adjustment basis and corporate-action reconciliation","PASS",
          "Manually attested in configuration; retain the supporting source documentation with the project.")

## 9. Accepted extraction exceptions

The three named failures are recorded as accepted coverage exceptions, not global data-quality blockers. The notebook checks whether each has a loaded price file. RAYMONDL is not automatically excluded if usable data exists; its extraction log and actual observations must be reconciled.

In [12]:
exception_rows=[]
price_ids=set(prices["ticker"].dropna().astype(str).str.strip().unique()) if len(prices) else set()
for x in sorted(ACCEPTED_EXTRACTION_EXCEPTIONS):
    files=prices.loc[prices["ticker"].astype(str).str.strip()==x] if len(prices) else pd.DataFrame()
    exception_rows.append({"ticker":x,"price_rows":len(files),
                           "present_in_price_data":x in price_ids,
                           "treatment":"ACCEPTED_EXCEPTION; exclude only if usable history is insufficient"})
exceptions_df=pd.DataFrame(exception_rows)
display(exceptions_df)

# Any other price extraction failure should not be silently accepted.

,ticker,price_rows,present_in_price_data,treatment
0,360ONE IN,0,False,ACCEPTED_EXCEPTION; exclude only if usable his...
1,RAYMONDL IN,0,False,ACCEPTED_EXCEPTION; exclude only if usable his...
2,SANOFICO IN,0,False,ACCEPTED_EXCEPTION; exclude only if usable his...


## 10. Fundamentals gate (only if fundamental strategies are enabled)

The current plan is price-only technical strategies, so fundamentals are not a prerequisite. If fundamental strategies are enabled, the data must have reliable public-availability timestamps and revision history. Fiscal period-end dates alone are insufficient.

In [13]:
fundamentals_pit_ok=False
if ENABLE_FUNDAMENTAL_STRATEGY:
    if fundamentals is None:
        issue("BLOCKER","fundamentals","FUNDAMENTALS_DATASET_MISSING","Fundamental strategy enabled but no fundamentals file found.")
    else:
        pub=find_col(fundamentals,["Announcement_Date","Publication_Date","Published_At","Available_At","Filing_Date"])
        rev=find_col(fundamentals,["Revision_Date","Revised_At","Updated_At","Restatement_Date"])
        if pub is None:
            issue("BLOCKER","fundamentals","NO_PUBLICATION_TIMESTAMP",
                  "Fundamental strategy enabled but no public availability field exists.")
        if rev is None:
            issue("BLOCKER","fundamentals","NO_REVISION_HISTORY",
                  "Fundamental strategy enabled but revision/restatement history is unavailable.")
        fundamentals_pit_ok=(pub is not None and rev is not None)
else:
    check("fundamentals","Fundamentals excluded from initial strategy","PASS",
          "Initial strategy is price-only; fundamentals are not used as signals or filters.")

## 11. Final readiness decision

A strategy is `READY_FOR_ENGINE_DEVELOPMENT` when the required data interfaces are structurally validated and unresolved evidence gaps are explicitly isolated. It is not `READY_FOR_PERFORMANCE_CLAIMS` until historical membership, price adjustments and execution assumptions are fully verified.

This notebook deliberately separates:
- engine-development readiness,
- backtest-result credibility,
- fundamental-strategy readiness.

In [14]:
issues_df=pd.DataFrame(issues)
checks_df=pd.DataFrame(checks)

# Blocking conditions for the initial price-only engine.
blocker_areas={"prices","mapping","membership","corporate_actions","features"}
blockers=issues_df[(issues_df["severity"]=="BLOCKER") & (issues_df["area"].isin(blocker_areas))] if len(issues_df) else pd.DataFrame()
fails=issues_df[(issues_df["severity"]=="FAIL") & (issues_df["area"].isin(blocker_areas))] if len(issues_df) else pd.DataFrame()

# A conservative decision: exact duplicate groups need resolution too; review flags remain non-blocking only if explicitly classified.
has_conflicting_dups = (issues_df["code"]=="CONFLICTING_DUPLICATE_PRICE_KEYS").any() if len(issues_df) else False
has_exact_dups = (issues_df["code"]=="EXACT_DUPLICATE_PRICE_KEYS").any() if len(issues_df) else False

if len(blockers) or len(fails) or has_conflicting_dups or has_exact_dups:
    engine_status="NOT_READY"
else:
    engine_status="READY_FOR_ENGINE_DEVELOPMENT"

# Credibility requires all manual historical evidence gates.
credibility_status=("READY_FOR_STRATEGY_BACKTEST_VALIDATION"
                    if MEMBERSHIP_EFFECTIVE_DATES_VERIFIED and PRICE_ADJUSTMENT_BASIS_VERIFIED
                    and CORPORATE_ACTION_RECONCILIATION_COMPLETED and FEATURE_DEFINITIONS_CONFIRMED
                    else "NOT_YET_CERTIFIED")

fundamental_status=("READY_FOR_FUNDAMENTAL_STRATEGIES" if fundamentals_pit_ok
                    else "NOT_READY_OR_NOT_ENABLED")

readiness=pd.DataFrame([
    {"gate":"Price-only engine development","status":engine_status,
     "meaning":"Whether validated data can safely be wired into the engine without unresolved structural blockers."},
    {"gate":"Credible historical strategy results","status":credibility_status,
     "meaning":"Requires verified effective membership, corporate-action/adjustment basis and feature definitions."},
    {"gate":"Fundamental strategies","status":fundamental_status,
     "meaning":"Requires point-in-time public-availability timestamps and revision history."}
])
display(readiness)
display(checks_df)
display(issues_df.sort_values(["severity","area","code"]) if len(issues_df) else issues_df)

,gate,status,meaning
0,Price-only engine development,NOT_READY,Whether validated data can safely be wired int...
1,Credible historical strategy results,NOT_YET_CERTIFIED,"Requires verified effective membership, corpor..."
2,Fundamental strategies,NOT_READY_OR_NOT_ENABLED,Requires point-in-time public-availability tim...


,area,check,status,detail,evidence
0,prices,No conflicting duplicate ticker/date keys,PASS,No conflicting duplicate groups found.,None
1,prices,No exact duplicate ticker/date keys,PASS,No exact duplicate groups found.,None
2,prices,OHLC ordering,PASS,No OHLC ordering violations among complete OHL...,None
3,fundamentals,Fundamentals excluded from initial strategy,PASS,Initial strategy is price-only; fundamentals a...,None


,severity,area,code,detail,evidence
8,BLOCKER,corporate_actions,CORPORATE_ACTION_RECONCILIATION_INCOMPLETE,Corporate actions and large price discontinuit...,None
7,BLOCKER,corporate_actions,PRICE_ADJUSTMENT_BASIS_UNVERIFIED,Vendor documentation has not established wheth...,None
6,BLOCKER,features,FEATURE_DEFINITIONS_NOT_ATTESTED,"Indicator formulas, initialization, price basi...",None
2,BLOCKER,mapping,MAPPING_SCHEMA_NOT_RECOGNIZED,Could not detect membership identifier and map...,None
4,BLOCKER,membership,MEMBERSHIP_EFFECTIVE_DATE_ATTESTATION_MISSING,Historical constituent effective dates have no...,None
3,BLOCKER,membership,SPARSE_SNAPSHOTS_NOT_EFFECTIVE_DATES,Only 20 snapshots. Set MEMBERSHIP_EFFECTIVE_DA...,None
5,REVIEW,features,FEATURE_MISMATCHES_OR_DEFINITION_DIFFERENCES,One or more supplied features differ from inde...,None
0,REVIEW,prices,EXTREME_DAILY_MOVES_UNRECONCILED,47 moves exceed 50%; reconcile with corporate ...,None
1,REVIEW,prices,PRICE_GAPS_UNRECONCILED,155 inter-observation gaps exceed 7 calendar d...,None


## 12. Export final validation reports

The output directory is separate from the source data. Preserve these reports alongside the code version used to produce them.

In [15]:
def safe_csv(df,name):
    if df is None: df=pd.DataFrame()
    df.to_csv(REPORT_ROOT/name,index=False)

safe_csv(issues_df,"final_issues.csv")
safe_csv(checks_df,"final_checks.csv")
safe_csv(readiness,"final_readiness.csv")
safe_csv(exceptions_df,"accepted_extraction_exceptions.csv")
safe_csv(mapping_coverage if "mapping_coverage" in globals() else pd.DataFrame(),"final_mapping_coverage.csv")
safe_csv(feature_comparison_df if "feature_comparison_df" in globals() else pd.DataFrame(),"final_feature_comparison.csv")
safe_csv(extreme,"final_extreme_returns.csv")
safe_csv(gaps,"final_price_gaps.csv")
safe_csv(conflicting_keys,"final_conflicting_duplicate_keys.csv")
safe_csv(exact_keys,"final_exact_duplicate_keys.csv")

summary={
    "generated_utc":datetime.now(timezone.utc).isoformat(),
    "data_root":str(DATA_ROOT),
    "price_files":len(price_paths),
    "price_rows":len(prices),
    "price_securities":int(prices["ticker"].nunique()) if len(prices) else 0,
    "membership_rows":len(membership),
    "membership_snapshots":int(snapshot_counts.shape[0]) if "snapshot_counts" in globals() else None,
    "mapping_rows":len(mapping),
    "duplicate_key_groups":len(signatures),
    "exact_duplicate_groups":len(exact_keys),
    "conflicting_duplicate_groups":len(conflicting_keys),
    "extreme_return_flags":len(extreme),
    "calendar_gap_flags":len(gaps),
    "accepted_extraction_exceptions":sorted(ACCEPTED_EXTRACTION_EXCEPTIONS),
    "engine_development_status":engine_status,
    "historical_results_status":credibility_status,
    "fundamental_strategy_status":fundamental_status,
    "issues_by_severity":issues_df["severity"].value_counts().to_dict() if len(issues_df) else {},
    "manual_evidence_flags":{
        "membership_effective_dates_verified":MEMBERSHIP_EFFECTIVE_DATES_VERIFIED,
        "price_adjustment_basis_verified":PRICE_ADJUSTMENT_BASIS_VERIFIED,
        "corporate_action_reconciliation_completed":CORPORATE_ACTION_RECONCILIATION_COMPLETED,
        "feature_definitions_confirmed":FEATURE_DEFINITIONS_CONFIRMED
    }
}
with open(REPORT_ROOT/"final_validation_summary.json","w") as f:
    json.dump(summary,f,indent=2,default=str)

print(json.dumps(summary,indent=2,default=str))
print("Reports written to:",REPORT_ROOT)

{
  "generated_utc": "2026-11-06T13:05:44.702110+00:00",
  "data_root": "/home/rishi/Documents/vscode/BacktestEngine/dataextracted",
  "price_files": 859,
  "price_rows": 1798332,
  "price_securities": 858,
  "membership_rows": 9857,
  "membership_snapshots": 20,
  "mapping_rows": 9857,
  "duplicate_key_groups": 0,
  "exact_duplicate_groups": 0,
  "conflicting_duplicate_groups": 0,
  "extreme_return_flags": 47,
  "calendar_gap_flags": 155,
  "accepted_extraction_exceptions": [
    "360ONE IN",
    "RAYMONDL IN",
    "SANOFICO IN"
  ],
  "engine_development_status": "NOT_READY",
  "historical_results_status": "NOT_YET_CERTIFIED",
  "fundamental_strategy_status": "NOT_READY_OR_NOT_ENABLED",
  "issues_by_severity": {
    "BLOCKER": 6,
    "REVIEW": 3
  },
  "manual_evidence_flags": {
    "membership_effective_dates_verified": false,
    "price_adjustment_basis_verified": false,
    "corporate_action_reconciliation_completed": false,
    "feature_definitions_confirmed": false
  }
}
Reports

## 13. Decision rules

- Do not change a manual evidence flag to `True` without retaining its supporting source documentation.
- Do not automatically delete exact duplicate observations. First verify they are duplicate extraction artifacts, then resolve them in a clean layer with provenance.
- Conflicting duplicate values must be reconciled against the vendor/source.
- Do not infer daily constituent membership from sparse snapshots.
- The three accepted extraction exceptions are coverage exclusions, not a reason to fail the entire dataset. Keep their treatment explicit.
- A `READY_FOR_ENGINE_DEVELOPMENT` result permits building and testing the software architecture. It does not certify profitable or unbiased backtest results.
- A credible historical backtest still needs point-in-time membership, corporate-action correctness, realistic next-session execution, costs, slippage and out-of-sample testing.